# Telecom Customer Churn Prediction Pipeline

## Objective

The objective of this notebook is to build an end-to-end prediction pipeline using the trained machine learning model.

The pipeline performs the following tasks:

- Load the saved machine learning model.
- Load the saved StandardScaler.
- Accept customer information.
- Apply the same preprocessing used during model training.
- Predict whether the customer is likely to churn.
- Calculate churn probability.
- Categorize customers into Low, Medium, or High Risk.
- Generate business recommendations based on customer characteristics.

This notebook demonstrates how a trained machine learning model can be converted into a reusable business decision-support system.

# Libraries

In [4]:
import pandas as pd
import numpy as np

import joblib
import json

# Load Saved Models

In [7]:
logistic_model = joblib.load("../models/logistic_regression.pkl")

random_forest_model = joblib.load("../models/random_forest.pkl")

scaler = joblib.load("../models/scaler.pkl")

print("Models Loaded Successfully!")

Models Loaded Successfully!


In [9]:
# ============================================================
# Load Best Hyperparameters
# ============================================================

with open("../models/best_parameters.json","r") as f:
    best_parameters = json.load(f)

print(best_parameters)

{'Logistic Regression': {'C': 100, 'solver': 'liblinear'}, 'Random Forest': {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}}


In [11]:
print("=" * 50)
print("Telecom Customer Churn Prediction System")
print("=" * 50)

print("\nLoaded Models")

print("- Logistic Regression")

print("- Random Forest")

print("\nBest Parameters")

print(best_parameters)

Telecom Customer Churn Prediction System

Loaded Models
- Logistic Regression
- Random Forest

Best Parameters
{'Logistic Regression': {'C': 100, 'solver': 'liblinear'}, 'Random Forest': {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}}


# Building the Prediction Pipeline

### Load Feature Names

In [19]:
X_train=pd.read_csv('../Data/processed/train_data.csv')
feature_columns=X_train.drop('Churn',axis=1).columns.tolist()
print("Number of Features :", len(feature_columns))
print(feature_columns)

Number of Features : 30
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


### Create Empty Customer Record

In [25]:
customer = pd.DataFrame(
    np.zeros((1, len(feature_columns))),
    columns=feature_columns
)

customer.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,MultipleLines_No phone service,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# **Prediction Function**

In [30]:
def predict_churn(
    gender,
    senior_citizen,
    partner,
    dependents,
    tenure,
    phone_service,
    multiple_lines,
    internet_service,
    online_security,
    online_backup,
    device_protection,
    tech_support,
    streaming_tv,
    streaming_movies,
    contract,
    paperless_billing,
    payment_method,
    monthly_charges,
    total_charges,
    model=logistic_model
):
    # Create empty dataframe with training columns
    customer = pd.DataFrame(
        np.zeros((1, len(feature_columns))),
        columns=feature_columns
    )
     # =====================================================
    # Label Encoding (same as Notebook 3)
    # =====================================================

    customer["gender"] = 1 if gender=="Male" else 0
    customer["SeniorCitizen"] = senior_citizen
    customer["Partner"] = 1 if partner=="Yes" else 0
    customer["Dependents"] = 1 if dependents=="Yes" else 0
    customer["PhoneService"] = 1 if phone_service=="Yes" else 0
    customer["PaperlessBilling"] = 1 if paperless_billing=="Yes" else 0

    # =====================================================
    # Numerical Features
    # =====================================================

    customer["tenure"] = tenure
    customer["MonthlyCharges"] = monthly_charges
    customer["TotalCharges"] = total_charges
     # =====================================================
    # One Hot Encoding
    # =====================================================

    # Multiple Lines
    if multiple_lines=="Yes":
        customer["MultipleLines_Yes"]=1
    elif multiple_lines=="No phone service":
        customer["MultipleLines_No phone service"]=1

    # Internet Service
    if internet_service=="Fiber optic":
        customer["InternetService_Fiber optic"]=1
    elif internet_service=="No":
        customer["InternetService_No"]=1

    # Online Security
    if online_security=="Yes":
        customer["OnlineSecurity_Yes"]=1
    elif online_security=="No internet service":
        customer["OnlineSecurity_No internet service"]=1

    # Online Backup
    if online_backup=="Yes":
        customer["OnlineBackup_Yes"]=1
    elif online_backup=="No internet service":
        customer["OnlineBackup_No internet service"]=1

    # Device Protection
    if device_protection=="Yes":
        customer["DeviceProtection_Yes"]=1
    elif device_protection=="No internet service":
        customer["DeviceProtection_No internet service"]=1

    # Tech Support
    if tech_support=="Yes":
        customer["TechSupport_Yes"]=1
    elif tech_support=="No internet service":
        customer["TechSupport_No internet service"]=1

    # Streaming TV
    if streaming_tv=="Yes":
        customer["StreamingTV_Yes"]=1
    elif streaming_tv=="No internet service":
        customer["StreamingTV_No internet service"]=1

    # Streaming Movies
    if streaming_movies=="Yes":
        customer["StreamingMovies_Yes"]=1
    elif streaming_movies=="No internet service":
        customer["StreamingMovies_No internet service"]=1

    # Contract
    if contract=="One year":
        customer["Contract_One year"]=1
    elif contract=="Two year":
        customer["Contract_Two year"]=1

    # Payment Method
    if payment_method=="Credit card (automatic)":
        customer["PaymentMethod_Credit card (automatic)"]=1
    elif payment_method=="Electronic check":
        customer["PaymentMethod_Electronic check"]=1
    elif payment_method=="Mailed check":
        customer["PaymentMethod_Mailed check"]=1
    # =====================================================
    # Scale Numerical Features
    # =====================================================

    numerical_cols = [
        "tenure",
        "MonthlyCharges",
        "TotalCharges"
    ]

    customer[numerical_cols] = scaler.transform(
        customer[numerical_cols]
    )
     # =====================================================
    # Prediction
    # =====================================================

    prediction = model.predict(customer)[0]

    probability = model.predict_proba(customer)[0][1]
     # =====================================================
    # Risk Level
    # =====================================================

    if probability < 0.30:
        risk = "LOW"

    elif probability < 0.70:
        risk = "MEDIUM"

    else:
        risk = "HIGH"
    # =====================================================
    # Business Recommendation
    # =====================================================

    recommendations = []

    if contract=="Month-to-month":
        recommendations.append("Offer long-term contract discount")

    if monthly_charges > 70:
        recommendations.append("Provide personalized pricing offer")

    if tech_support=="No":
        recommendations.append("Promote Tech Support service")

    if online_security=="No":
        recommendations.append("Promote Online Security service")

    if risk=="HIGH":
        recommendations.append("Contact customer immediately")

    return {
        "Prediction":"Churn" if prediction==1 else "No Churn",
        "Probability":round(probability*100,2),
        "Risk Level":risk,
        "Recommendations":recommendations
    }


## Testing the Function

In [33]:
result = predict_churn(

    gender="Male",
    senior_citizen=0,
    partner="Yes",
    dependents="No",
    tenure=12,
    phone_service="Yes",
    multiple_lines="Yes",
    internet_service="Fiber optic",
    online_security="No",
    online_backup="Yes",
    device_protection="No",
    tech_support="No",
    streaming_tv="Yes",
    streaming_movies="Yes",
    contract="Month-to-month",
    paperless_billing="Yes",
    payment_method="Electronic check",
    monthly_charges=89.50,
    total_charges=1074.00

)

result

{'Prediction': 'Churn',
 'Probability': 87.6,
 'Risk Level': 'HIGH',
 'Recommendations': ['Offer long-term contract discount',
  'Provide personalized pricing offer',
  'Promote Tech Support service',
  'Promote Online Security service',
  'Contact customer immediately']}

In [35]:
print("="*55)
print("      TELECOM CUSTOMER CHURN PREDICTION")
print("="*55)

print(f"Prediction      : {result['Prediction']}")
print(f"Probability     : {result['Probability']} %")
print(f"Risk Level      : {result['Risk Level']}")

print("\nBusiness Recommendations")

for i, rec in enumerate(result["Recommendations"],1):
    print(f"{i}. {rec}")

print("="*55)

      TELECOM CUSTOMER CHURN PREDICTION
Prediction      : Churn
Probability     : 87.6 %
Risk Level      : HIGH

Business Recommendations
1. Offer long-term contract discount
2. Provide personalized pricing offer
3. Promote Tech Support service
4. Promote Online Security service
5. Contact customer immediately
